In [1]:
import torch
from train import train
from lofi_model import LofiModel
from dataset import MidiDataset
import pretty_midi
import numpy as np
from config import *
print(torch.__version__)
print(torch.version.cuda)

c:\Users\Hyperbook\Desktop\STUDIA\SEM III\PROJEKT ZESPOLOWY\venv\Lib\site-packages\pretty_midi\instrument.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


2.7.1+cu118
11.8


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
model = LofiModel(device)
model.to(device)
# model.load_state_dict(torch.load("./saved_models/multitrack LSTM-VAE (1 min).pth")) # BEST MODEL
model.load_state_dict(torch.load(r"C:\Users\Hyperbook\Desktop\STUDIA\SEM III\PROJEKT ZESPOLOWY\model\saved_models\lofi-model_epoch2.pth")) 
model.to(device)
model.eval()

LofiModel(
  (encoder_lstm): LSTM(305, 512, batch_first=True)
  (fc_mu): Linear(in_features=512, out_features=512, bias=True)
  (fc_logvar): Linear(in_features=512, out_features=512, bias=True)
  (decoder_lstm): LSTM(512, 512, batch_first=True)
  (fc_output): Linear(in_features=512, out_features=305, bias=True)
)

## Generation

In [3]:
model.eval()
generated_tensor, midi_file = model.generate()#, save_path="generated/genearated_xyz.mid")


✅ MIDI saved to: generated/generated.mid


In [4]:
g = generated_tensor[0][2]
g

tensor([-1.0512, -2.7290, -5.9286,  ..., -8.7927, -8.7927, -8.7927])

In [5]:
generated_tensor.shape

torch.Size([5, 61, 1920])

In [6]:
dataset_dir = r"C:\Users\Hyperbook\Desktop\STUDIA\SEM III\PROJEKT ZESPOLOWY\dataset\transformed_dataset"
dataset = MidiDataset(dataset_dir, verbose=True)
# for i in range(10,15):
i = 12
original_sequence_tensor, original_length_val, bpm, filename = dataset[i]
original_length_val = torch.tensor(original_length_val, dtype=torch.int64)
original_sequence_tensor = original_sequence_tensor.unsqueeze(0).to(device)  # -> (B, T, I, P)
x_flat = original_sequence_tensor.permute(0, 3, 1, 2).contiguous() # -> (B, T, I, P)
x_flat = x_flat.view(1, original_length_val, -1) # -> (B, T, I*P)
x_flat = x_flat.to(device)  # -> (B, T, I*P)

In [7]:
mu, logvar = model.encoder(x_flat, original_length_val.unsqueeze(0).to(device))
z = model.reparameterize(mu, logvar)

AttributeError: 'LofiModel' object has no attribute 'encoder'

In [ ]:
model.eval()
generated_tensor, midi = model.generate(1, MAX_SEQ_LEN,  z_sample= z, save_path="generated/genearated_xyz.mid")

tensor.shape=torch.Size([5, 61, 640]), tensor.dtype=torch.float32, tensor.device=device(type='cuda', index=0)
MIDI saved to generated/genearated_xyz.mid
